# **Annotate a molecular dynamics description with LLM 📝**

### 🎯 Objectives:

- Annotate a description of a datasats or a Materials and Methods section of a molecular dynamics study.
- Identify key experimental procedures, parameters, and computational tools like MOLECULES (`MOL`), FORCEFIELDS AND MODELS (`FFM`), SIMULATION TIMES (`STIME`), SIMULATION TEMPERATURE (`STEMP`), SOFTWARE NAMES (`SOFTNAME`) and SOFTWARE VERSIONS (`SOFTVERS`).
- Control the output format of the annotations with `instructor` framework to reduce hallucinations.
- Visualize the annotation results for clarity and verification.


## **Load libraries and setup**

In [15]:
import os
from pathlib import Path

from dotenv import load_dotenv

from mdner_llm.annotations.visualize_annotations import visualize_llm_annotation
from mdner_llm.core.extract_entities_with_llm import (
    add_guidelines_and_examples_to_prompt,
    annotate_with_llm_and_framework,
)
from mdner_llm.models.entities import ListOfEntities


In [3]:
%load_ext watermark
%watermark

Last updated: 2026-06-26T22:56:03.909145+02:00

Python implementation: CPython
Python version       : 3.13.13
IPython version      : 8.13.2

Compiler    : Clang 22.1.3 
OS          : Darwin
Release     : 25.4.0
Machine     : arm64
Processor   : arm
CPU cores   : 15
Architecture: 64bit



## **Step 1: Define the text to annotate**


In [4]:
TEXT_TO_ANNOTATE = """Molecular Dynamics Simulations

MD simulations were performed using GROMACS 4.0.7 software [17] with the OPLS-AA
force-field [18].L33 and P33 forms of β3 were soaked in a rhombic dodecahedral
simulation box with 60,622 TIP3P water molecules and 28 Na+ ions.
The distance between any atom of the protein and the box edges was set to at least 10 Å.
The total energy of the system was minimized twice (before and after the addition of the
ions) with a steepest descent algorithm. MD simulations were run under the NPT
thermodynamic ensemble and periodic boundary conditions were applied in all directions.
We used the weak coupling algorithm of Berendsen [19] to maintain the system at a
constant physiological temperature of 310 K using a coupling constant of 0.1 ps (protein
and water ions separately). Pressure was held constant using the Berendsen algorithm
[19] at 1 atm with a coupling constant of 1 ps. Water molecules were kept rigid using
the SETTLE algorithm [20]. All other bond lengths were constrained with the LINCS
algorithm [21], allowing a 2 fs time step. We used a short-range coulombic and van der
Waals cut-off of 10 Å and calculated the long-range electrostatic interactions using
the smooth particle mesh Ewald (PME) algorithm [22], [23] with a grid spacing of 1.2 Å
and an interpolation order of 4. The neighbor list was updated every 10 steps. After a
1 ns equilibration (with position restraints on the protein), each system was simulated
for 50 ns. For both systems, five 50 ns simulations were performed (using different
initial velocities) and one was extended until 100 ns for a total simulation time of
300 ns. Molecular conformations were saved every 100 ps for further analysis.
"""

## **Step 2: Select the model**


But before, we retrieve the API key for the openrouter API from the environment variables. Make sure to set the `OPENROUTER_API_KEY` environment variable before running this cell.


In [5]:
# Set the openrouter API key
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")

In [12]:
# Choices:
# Docs: https://openrouter.ai/models
# We will use "openai/gpt-5.2" for annotation
SELECTED_MODEL = "openai/gpt-5.2"
TEMPERATURE = 1.0
PROVIDER = "openai"

## **Step 3: Load the prompt template**


In [10]:


prompt_path = Path("../docs/prompt_template.md")
prompt = prompt_path.read_text(encoding="utf-8")
prompt_filled = add_guidelines_and_examples_to_prompt(
    prompt, Path("../docs/annotation_rules.md"), Path("../docs/few_shot_examples.md")
)
print(f"Prompt loaded from {prompt_path}:")
print(prompt_filled)


Prompt loaded from ../docs/prompt_template.md:
# Named-Entity Recognition task

## Role definition

You are a highly specialized **Named Entity Recognition (NER) engine** dedicated to extracting entities from Molecular Dynamics (MD) simulation dataset descriptions or research articles.
Your SOLE function is to analyze the input text and output a valid JSON object containing only the extracted entities and their categories.

## Annotation rules

### Molecule: MOL

#### Definition

The **MOL** entity covers molecular compounds, including simple molecules, ions, nucleic acids, proteins, lipids, sugars, polymers, and complexes.
MOL entities should be uniquely identifiable.

#### Rules

1. Exclude whitespace and punctuation marks around entities
2. The molecule name should be at least 2 characters long
3. Treat singular and plural forms equivalently and annotate both (e.g., `alkane` and `alkanes`)
4. Annotate chemical formulas and abbreviations separately from full molecule names.
5. Includ

## **Step 4: Annotate the text**


In [16]:
response, inference_metadata = annotate_with_llm_and_framework(
    framework="instructor",
    text_to_annotate=TEXT_TO_ANNOTATE,
    model=SELECTED_MODEL,
    provider=PROVIDER,
    temperature=TEMPERATURE,
    api_key=API_KEY,
    prompt=prompt_filled,
    response_model=ListOfEntities,
    max_retries=3,
)
print(
    f"Annotation by {SELECTED_MODEL} completed in "
    f"{inference_metadata['usage']['inference_time_sec']:.2} "
    f"seconds with cost ${inference_metadata['usage']['cost']:.4f}."
)
response

2026-06-26 23:08:36.616 | DEBUG    | mdner_llm.core.extract_entities_with_llm:annotate_with_llm_and_framework:337 - Starting annotation with model openai/gpt-5.2 using instructor.


Annotation by openai/gpt-5.2 completed in 1.2e+01 seconds with cost $0.0163.


ListOfEntities(entities=[SoftwareName(category='SOFTNAME', text='GROMACS', score=None), SoftwareVersion(category='SOFTVERS', text='4.0.7', score=None), ForceFieldModel(category='FFM', text='OPLS-AA', score=None), Molecule(category='MOL', text='L33', score=None), Molecule(category='MOL', text='P33', score=None), Molecule(category='MOL', text='β3', score=None), ForceFieldModel(category='FFM', text='TIP3P', score=None), Molecule(category='MOL', text='Na+', score=None), SimulationTemperature(category='STEMP', text='310 K', score=None), SimulationTime(category='STIME', text='50 ns', score=None), SimulationTime(category='STIME', text='50 ns', score=None), SimulationTime(category='STIME', text='100 ns', score=None), SimulationTime(category='STIME', text='300 ns', score=None)])

## Step 5: Visualize the annotation results


In [17]:
visualize_llm_annotation(response, TEXT_TO_ANNOTATE)

🧐 VISUALIZATION OF ENTITIES 
